<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Introduction to Docker

This notebook introduces Docker and the specific features used in Duckietown. For a broader overview, consult [Docker's official Get Started guide](https://docs.docker.com/get-started/overview/), and see the [Duckietown Introduction to Containerization lecture](https://app-na1.hubspotdocuments.com/documents/8795519/view/717428738?accessId=475a8d) for additional insight.

## Why Docker?

In other learning experiences (LXs), such as the [Linux and Networking LX](https://github.com/duckietown/lx-dd-linux-and-networking), you learned how to run programs, inspect processes, and connect to a Duckiedrone. Running your own robotics software introduces another challenge: the program needs the right environment.

For example, a camera-processing script may depend on a particular Python version and several image-processing libraries. Copying the script from your base station to a Duckiedrone does not copy those dependencies. Installing them directly on the Duckiedrone can also create conflicts with software already there.

Docker provides operating-system-level virtualization, commonly called _containerization_. Containers isolate user-space instances so that programs perceive a dedicated environment while sharing the host kernel. This approach yields lightweight, reproducible deployments, which are useful on hardware such as the Raspberry Pi used in Duckietown.

In other words, containerization allows the existence of multiple isolated user-space instances called containers. These containers may look like real computers from the point of view of programs running in them.

Containers provide separate views of resources such as process identifiers, filesystem mounts, and networking. They still share the host's processor and memory; resource limits depend on the container's configuration.

This notebook explains what a container is, how images define its software environment, and how Docker manages it. These distinctions help you interpret a container's files, processes, and execution environment.

## What is a container?

Consider two applications that process Duckiedrone camera images. One requires an older version of an image-processing library; the other requires a newer version. Installing both into the same software environment may be difficult.

With containers, each application can have its own files, libraries, and installed tools. Both can run on the same computer without requiring the same library version.

From inside a container, a shell can look familiar: commands such as `ls`, `cd`, and `cat` still work. However, the filesystem you explore belongs to the container's environment. Installing a Python library there does not normally install it into the host's Python environment.

The __host__ is the computer providing the resources on which the container runs. Containers use its processor, memory, and kernel; they do not provide their own additional computing capacity.

Docker uses Linux __namespaces__ to give processes separate views of resources such as filesystem mounts, process identifiers, and networking. The degree of separation depends on the container's configuration. For example, a directory can be deliberately shared between the host and a container. For details, see [Docker's underlying technology](https://docs.docker.com/get-started/docker-overview/#the-underlying-technology).

## How is a container different from a virtual machine?

A virtual machine also provides an environment in which to run software. The important difference is what that environment includes.

A __virtual machine (VM)__ runs a guest operating system with its own kernel. Linux containers share the kernel of their Linux host, while keeping their application environments separate. [Figure 1](#figure-1) compares these two architectures.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    virtual machines                           Linux containers
    +-----------------+ +-----------------+    +-----------------+ +-----------------+
    | application A   | | application B   |    | application A   | | application B   |
    | libraries       | | libraries       |    | libraries       | | libraries       |
    | guest kernel    | | guest kernel    |    +-----------------+ +-----------------+
    +-----------------+ +-----------------+    |         shared Linux kernel         |
    |        virtualization layer         |    +-------------------------------------+
    +-------------------------------------+
  </pre>
  <figcaption>
    Figure 1: Each VM has its own guest kernel, while Linux containers share the host kernel.
  </figcaption>
</figure>

Sharing a kernel generally reduces the overhead of running separate environments. This is useful on a Duckiedrone, where memory and processing power must also support sensing, communication, and control.

An Ubuntu container can therefore provide Ubuntu command-line tools and libraries without booting a separate Ubuntu kernel. On systems such as macOS, Linux containers run inside a Linux VM; they share that VM's kernel. See Docker's [What is a container?](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/) guide.

## How does Docker work?

Images contain filesystem layers and configuration; containers are created from images. These concepts have distinct roles:

- Image: build-time artifact

- Layer: incremental filesystem change inside an image

- Container: run-time instance of an image

Docker images are _build-time_ artifacts while Docker containers are _run-time_ constructs.

In other words, a Docker image is static, like a `.zip` or `.iso` file. A container, on the other hand, starts from a static image but as it is used, files and configurations might change.

An image usually starts from a __base image__, which can itself contain several layers. For example, an Ubuntu base image supplies Linux user-space tools and libraries.

Each layer added on top of the base image represents a change to the layers below. Docker translates this sequence of changes to a file system that the container can then use.

Changing a file copied into an image can invalidate that build step and the steps after it. When pulling the rebuilt image, Docker reuses layers already stored locally and downloads any missing layers.

## Images define the environment and containers use it

Before starting our camera-processing application, Docker needs a description of the files and software it should make available.

An __image__ is the read-only template used to create a container. For our example, it could contain Python, the image-processing library, and the application code.

A __container__ is an instance created from that image. One image can be used to create several containers, each with its own configuration and writable filesystem layer.

For example, we could use the same image to create:

- `lx-docker-camera-a`, which processes a recording from one experiment.

- `lx-docker-camera-b`, which processes a recording from another experiment.

Both begin with the same application and dependencies. A file written into the writable layer of the first container does not automatically appear in the second or change the original image.

Stopping a container preserves its writable layer. Removing the container deletes that layer. Persistent storage can keep experiment results independently of a container's lifetime. See the [Docker storage](https://docs.docker.com/engine/storage/) documentation.

### How is an image built?

A __Dockerfile__ records the instructions for building an image. The build could start from an existing Python image, install the required library, and copy in a processing script.

These filesystem changes would be stored in __layers__, as illustrated in [Figure 2](#figure-2).

<figure id="figure-2" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    illustrative image contents
    +-------------------------------------------------------------+
    | camera-processing script                                    |
    +-------------------------------------------------------------+
    | additional image-processing library                         |
    +-------------------------------------------------------------+
    | base image: Python and supporting Linux tools and libraries |
    +-------------------------------------------------------------+
  </pre>
  <figcaption>
    Figure 2: Image layers store filesystem changes over the base image.
  </figcaption>
</figure>

The base image may itself contain several layers. Docker combines them into the filesystem visible inside a container and can reuse layers already available locally. This reduces repeated downloads and storage when images share software. For additional detail, see Docker's [Understanding image layers](https://docs.docker.com/get-started/docker-concepts/building-images/understanding-image-layers/) guide.

## Registries make images available to other computers

Suppose we have built a processing image, and a teammate wants to use it.

A __registry__ stores and distributes images. [Docker Hub](https://hub.docker.com/) is a public registry used by Duckietown. __Pushing__ uploads an image to a registry; __pulling__ downloads it.

[Figure 3](#figure-3) shows the workflow for sharing an image through a registry.

<figure id="figure-3" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    build an image
      |
      v
    push it to a registry
      |
      v
    pull it on another computer
      |
      v
    create a container
  </pre>
  <figcaption>
    Figure 3: Sharing an image through a registry.
  </figcaption>
</figure>

Pulling an image does not run the application. It makes the image available so that Docker can create containers from it.

Packaging the software also does not remove hardware requirements. An image must support the target computer's processor architecture, and an application that reads a physical camera still needs access to that device.

### Docker Hub and Duckietown

[Figure 4](#figure-4) shows the structure of a Docker Hub image reference.

<figure id="figure-4" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    [namespace/]repository[:tag]
  </pre>
  <figcaption>
    Figure 4: Docker Hub image reference structure.
  </figcaption>
</figure>

The namespace and tag are optional. If omitted, Docker uses the `library` namespace for Docker Official Images and the `latest` tag. `latest` is an ordinary mutable tag chosen by the image publisher; it does not automatically identify the newest image version. See the [Docker image references](https://docs.docker.com/reference/cli/docker/image/tag/) documentation.

[Figure 5](#figure-5) gives an example of a Duckietown image reference.

<figure id="figure-5" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    duckietown/dt-core:ente-arm64v8
  </pre>
  <figcaption>
    Figure 5: Example Duckietown image reference.
  </figcaption>
</figure>

The image reference has three parts:

- The namespace is `duckietown`.

- The repository name is `dt-core`, giving the full repository name `duckietown/dt-core`.

- The tag is `ente-arm64v8`.

The tag names the Duckietown software distribution, `ente`, and the target central processing unit (CPU) architecture, `arm64v8`, in this example.

Duckietown publishes multiple image repositories under the `duckietown` Docker Hub namespace.

## Docker Engine manages images and containers

When you ask Docker to start a container, two components cooperate:

- The __Docker client__, usually the `docker` command-line interface, sends your request.

- The __Docker daemon__, `dockerd`, carries it out and manages resources such as images, containers, networks, and storage volumes.

They communicate through an __Application Programming Interface (API)__. Together, these components form Docker Engine's client-server architecture. See the [Docker Engine](https://docs.docker.com/engine/) documentation.

The client and daemon can run on different computers. For example, a client on your base station can connect to a daemon on the Duckiedrone. A request to start a container then starts it on the Duckiedrone. [Figure 6](#figure-6) shows this remote client-daemon connection.

<figure id="figure-6" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    base-station Docker client
      |
      | API request
      v
    Duckiedrone Docker daemon
      |
      | create and start a container
      v
    container on the Duckiedrone
  </pre>
  <figcaption>
    Figure 6: A base-station client requests a container on the Duckiedrone.
  </figcaption>
</figure>

A __Docker context__ stores the connection settings used to select a daemon. Therefore, the terminal in which you type a command does not, by itself, identify where Docker will perform the action. See the [Docker contexts](https://docs.docker.com/engine/manage-resources/contexts/) documentation.

Before creating practice resources, verify that the client reaches your local daemon. Names beginning with `lx-docker-` identify the exercise resources.

## Reflection

Before running the self-check, consider these questions:

1. Why can two containers use different Python libraries while sharing the same Linux kernel?

2. You create two containers from one image. Does writing a file in one container's writable layer change the image?

3. Your Docker client runs on the base station and connects to the Duckiedrone's daemon. Where does a new container run?

## Further reading

The [Duckietown Docker introduction](https://docs.duckietown.com/ente/duckietown-manual/70-developer-manual/basics/development/developer-basics-docker.html) connects these concepts to the tools and workflows used in Duckietown. Docker's [What is Docker?](https://docs.docker.com/get-started/docker-overview/) guide provides a broader platform overview.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()


## Summary and conclusions

Images package software and its dependencies in layers; containers run instances of those images while sharing the Docker host's kernel. The client asks a daemon to manage these resources, and registries distribute images. Next, [prepare a local practice environment](./2-docker-containers-images-and-safe-local-practice.ipynb) and check where your Docker commands will run.
